# Practice 2 / Lab 2: Người 3 - Hyperparameter và fine-tuning

Đọc README và docs giao việc trước khi chạy. Chọn GPU trong Runtime. Notebook này chưa có kết quả thực nghiệm. Chạy từng ô; không dùng smoke accuracy để báo cáo. Không reset runtime trước khi tải checkpoint/kết quả.

## 1. Chuẩn bị môi trường
Nếu chưa upload nền tảng lên GitHub, hãy tải ZIP vào Colab và giải nén, đổi thư mục đến repo; sau đó cài requirements. Ô dưới dành cho repo đã có code trên main.

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO = "https://github.com/kakal1090/lab2-pretrained-networks.git"
BRANCH = "main"  # Chỉ dùng sau khi nền tảng đã merge. Trước đó dùng nhánh đã push thật.
workspace = Path('/content') if Path('/content').exists() else Path.cwd()
repo_dir = workspace / 'lab2-pretrained-networks'
if not repo_dir.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, str(repo_dir)], check=True)
os.chdir(repo_dir)
assert Path('src/train.py').exists(), 'Repo chưa có nền tảng. Merge gói code trước hoặc chọn đúng nhánh.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU - chạy đầy đủ có thể chậm')
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## 2. Kiểm tra code và kiến trúc
ImageNet pretrained; lớp cuối đổi thành 10. Mode head chỉ học classifier. Đọc danh sách trainable để hiểu phần mình.

In [ ]:
from src.train import build_model
model = build_model('resnet18', 'head')
print(model)
print('Trainable:', [n for n,p in model.named_parameters() if p.requires_grad])
del model


## 3. Chạy thực nghiệm thật
Mỗi run-name chỉ được dùng một lần để tránh ghi đè. Nếu chạy lại dùng hậu tố _r2, ghi rõ trong báo cáo. Dữ liệu được tải tự động, phân chia 45.000/5.000; test chưa sử dụng. Mỗi model khởi tạo lại từ ImageNet.

### p3_lr_01

In [ ]:
subprocess.run([sys.executable] + ['-m', 'src.train', '--run-name', 'p3_lr_01', '--model', 'resnet18', '--mode', 'head', '--lr', '0.01', '--batch-size', '32', '--epochs', '5', '--seed', '42'], check=True)


### p3_batch64

In [ ]:
subprocess.run([sys.executable] + ['-m', 'src.train', '--run-name', 'p3_batch64', '--model', 'resnet18', '--mode', 'head', '--lr', '0.001', '--batch-size', '64', '--epochs', '5', '--seed', '42'], check=True)


### p3_finetune

In [ ]:
subprocess.run([sys.executable] + ['-m', 'src.train', '--run-name', 'p3_finetune', '--model', 'resnet18', '--mode', 'finetune', '--lr', '0.001', '--batch-size', '32', '--epochs', '5', '--seed', '42'], check=True)


## 4. Xem số liệu và TensorBoard
Chụp đủ 4 đồ thị, giữ tên run và trục epoch. So sánh với baseline của người 1; accuracy trong JSON là tỷ lệ 0-1.

In [ ]:
import json
RUNS = ['p3_lr_01', 'p3_batch64', 'p3_finetune']
for run in RUNS:
    print(run, json.loads((Path('results')/run/'summary.json').read_text()))


In [ ]:
%load_ext tensorboard
%tensorboard --logdir results


## 5. Chốt cấu hình rồi mới đánh giá test
Người 3: thay SELECTED_RUN theo validation, ghi lý do trước khi chạy. Không thử test từng cấu hình để chọn. Dùng checkpoint của best validation epoch, không train lại. Nếu file test_metrics đã có, đọc kết quả cũ.

In [ ]:
SELECTED_RUN = 'DIEN_RUN_DA_CHON_THEO_VALIDATION'
assert SELECTED_RUN in RUNS, 'Chọn tên run đã chạy dựa trên validation'
subprocess.run([sys.executable, '-m', 'src.evaluate', '--run-dir', f'results/{SELECTED_RUN}'], check=True)


## 6. Nhận xét của người thực hiện
Điền sau khi chạy: GPU/phiên bản, best epoch, train/val loss và accuracy, test cuối; đồ thị có overfitting không; cấu hình khác baseline ở đâu; hạn chế. Người 2 so sánh kiến trúc, người 3 phân tích từng tham số. Không giữ nguyên ô này để nộp.

## 7. Tải kết quả và giữ checkpoint
Gói submission chứa JSON/CSV/TXT và ảnh nếu bạn đã lưu trong figures. Gói checkpoint riêng KHÔNG đưa lên GitHub. Bạn phải tự chụp TensorBoard và thêm vào figures. Kết quả Colab có thể mất khi runtime reset.

In [ ]:
import zipfile
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
for run in RUNS:
    (Path('results')/run/'environment.txt').write_text(freeze)
submission = workspace / 'submission_person3.zip'
with zipfile.ZipFile(submission, 'w', zipfile.ZIP_DEFLATED) as z:
    for run in RUNS:
        folder=Path('results')/run
        for f in folder.rglob('*'):
            if f.is_file() and 'tensorboard' not in f.parts and f.suffix not in ('.pt','.pth'):
                z.write(f, str(f))
print(submission)
if Path('/content').exists():
    from google.colab import files
    files.download(str(submission))


In [ ]:
backup = workspace / 'checkpoints_person3.zip'
with zipfile.ZipFile(backup, 'w', zipfile.ZIP_DEFLATED) as z:
    for run in RUNS:
        f=Path('results')/run/'best.pt'
        if f.exists(): z.write(f, str(f))
print('Chỉ lưu cá nhân, không commit:', backup)
if Path('/content').exists():
    files.download(str(backup))


## 8. Nộp Pull Request
Tải notebook đã điền nhận xét và gói submission về máy; đặt kết quả vào repo; thêm báo cáo của mình, commit nhánh riêng và mở PR base main, reviewer kakal1090. Xem docs/GIT_WORKFLOW.md. Chờ trưởng nhóm duyệt.